# Jev (via OpenRouter): the same tests, plus a token-count analysis
Kaggle settings: **Accelerator = None** (CPU), **Internet = On**, and your OpenRouter key in
**Add-ons → Secrets** under the name `OPENROUTER_API_KEY` (allow this notebook to read it).
First run with `QUICK = True`. If everything passes, set it to `False` and use *Save Version → Save & Run All*.
Same texts, options and seeds as the Laya, Lev and CLM notebooks.

In [ ]:
!pip install -q statsmodels

In [ ]:
# 0. Configuration
import os, json, time, re, sys, platform, logging, zipfile, threading, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt

QUICK = True
MODEL = "typesafe/jev-1.13"                      # pinned version; each response reports the exact version that served it
URL = "https://openrouter.ai/api/v1/systemone"   # OpenRouter's System One API (same format as TypeSafe's)
GEN_URL = "https://openrouter.ai/api/v1/generation"  # per-call metadata on OpenRouter, including server-side time
MAX_OPTIONS = 255                                # documented limit of options per question
WORKERS = 4                                      # concurrent calls in the accuracy runs (latency is always measured one call at a time)

CFG = dict(
    seed=42,
    exp1_n_options=[2, 10, 50, 150, 255], exp1_n_examples=100, exp1_latency_repeats=30,
    exp2_n_options=[4, 10, 20, 50, 77],   exp2_n_examples=100, exp2_n_perms=5,
    exp2b_n_examples=100, exp4_per_intent=20, expR_repeats=10,
)
if QUICK:
    CFG.update(exp1_n_examples=5, exp1_latency_repeats=5, exp2_n_examples=5, exp2_n_perms=3,
               exp2b_n_examples=3, exp4_per_intent=2, expR_repeats=3)
RUN = dict(exp1=True, exp2=True, exp2b=True, exp3b=True, exp4=True, expR=True)

try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
assert API_KEY, "OpenRouter key not found (Kaggle Secrets or the OPENROUTER_API_KEY variable)"

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT = ROOT / "jev_bench"
for d in ["results", "logs", "figures"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)
RUN_ID = time.strftime("%Y%m%d-%H%M%S") + ("-quick" if QUICK else "")
SFX = "_quick" if QUICK else ""

log = logging.getLogger("jev_bench"); log.setLevel(logging.INFO); log.handlers.clear()
for h in [logging.FileHandler(OUT / "logs" / f"run_{RUN_ID}.log"), logging.StreamHandler(sys.stdout)]:
    h.setFormatter(logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")); log.addHandler(h)
log.info(f"run_id={RUN_ID} out={OUT} model={MODEL}")

In [ ]:
# 1. HTTP client, metadata and sanity check
_local = threading.local()
def session():
    if not hasattr(_local, "s"):
        _local.s = requests.Session()
        _local.s.headers.update({"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json",
                                 "X-Title": "system-one-architecture-bench"})
    return _local.s

GEN_RE = re.compile(r"gen-[A-Za-z0-9_-]{6,}")
def gen_id_of(js, headers):
    """OpenRouter call id: the response's id field or, failing that, any header containing gen-..."""
    v = js.get("id") if isinstance(js, dict) else None
    if isinstance(v, str) and v:
        return v
    for val in headers.values():
        m = GEN_RE.search(str(val))
        if m:
            return m.group(0)
    return None

RETRY = {408, 409, 425, 429, 500, 502, 503, 504, 520, 522, 524, 529}
def call(state, questions, timeout=120, max_attempts=6):
    """One call to Jev. Measures the wall time of the successful attempt and counts the attempts."""
    body = {"model": MODEL, "state": state, "questions": questions}
    err = None
    for attempt in range(1, max_attempts + 1):
        t = time.perf_counter()
        try:
            r = session().post(URL, json=body, timeout=timeout)
            ms = (time.perf_counter() - t) * 1000
        except requests.RequestException as e:
            err = repr(e)[:300]; time.sleep(min(30, 2 ** attempt)); continue
        if r.status_code == 200:
            js = r.json()
            return dict(ok=True, js=js, ms=ms, attempts=attempt, status=200, gen_id=gen_id_of(js, r.headers))
        err = f"HTTP {r.status_code}: {r.text[:300]}"
        if r.status_code not in RETRY:
            return dict(ok=False, status=r.status_code, error=err, ms=ms, attempts=attempt)
        wait = r.headers.get("Retry-After")
        time.sleep(float(wait) if wait and wait.replace(".", "", 1).isdigit() else min(30, 2 ** attempt + random.random()))
    return dict(ok=False, status=-1, error=err, attempts=max_attempts)

def usage(js):
    u = js.get("usage", {}) or {}
    return dict(input_tokens=u.get("input_tokens", u.get("prompt_tokens")),
                output_tokens=u.get("output_tokens", u.get("completion_tokens")), cost=u.get("cost"),
                served_model=js.get("model"))

def choice_q(criteria, instructions):
    return {"q": {"type": "choice", "instructions": instructions, "criteria": criteria}}

def answer(js, qid="q"):
    a = js["answers"][qid]
    return a["choice"], {k: float(v) for k, v in a["probabilities"].items()}, a.get("confidence")

def decimals(x):
    s = repr(float(x))
    return 0 if "." not in s or "e" in s else len(s.split(".")[1].rstrip("0"))

sanity_q = {"department": {"type": "choice", "instructions": "Which team should handle this?",
            "criteria": {"returns": "Exchanges, wrong or damaged items", "shipping": "Delivery status, delays, lost packages",
                         "billing": "Charges, invoices, payment problems"}}}
sanity_state = "My running shoes arrived in the wrong size. Can I swap them for a size 10?"
t0 = time.perf_counter()
r = call(sanity_state, sanity_q)
assert r["ok"], f"Call failed: {r.get('error')}"
raw = session().post(URL, json={"model": MODEL, "state": sanity_state, "questions": sanity_q}, timeout=120)
meta = dict(run_id=RUN_ID, quick=QUICK, cfg=CFG, model=MODEL, url=URL, workers=WORKERS,
            python=sys.version.split()[0], platform=platform.platform(),
            sanity_response=r["js"], sanity_ms=r["ms"], response_headers=dict(raw.headers),
            docs_reference={"department": "returns", "probability": 1.0})
(OUT / "results" / f"run_meta_{RUN_ID}.json").write_text(json.dumps(meta, indent=2, default=str))
ch, pr, conf = answer(r["js"], "department")
log.info(f"sanity: {ch} {pr} conf={conf} | {usage(r['js'])} | {r['ms']:.0f} ms")
assert ch == "returns", "Sanity check failed: the documentation example did not return 'returns'"

In [ ]:
# 2. Utilities: thread-safe JSONL logs, parallel execution, metrics
_wlock = threading.Lock()
def jdefault(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return str(o)

class JsonlLog:
    """One line per call. Keys already written are skipped (resume within a session)."""
    def __init__(self, name):
        self.path = OUT / "results" / f"{name}{SFX}.jsonl"
        self.done = set()
        if self.path.exists():
            for line in self.path.open():
                try: self.done.add(json.loads(line)["key"])
                except Exception: pass
    def write(self, rec):
        rec = {"run_id": RUN_ID, **rec}
        with _wlock:
            with self.path.open("a") as f: f.write(json.dumps(rec, default=jdefault) + "\n")
            self.done.add(rec["key"])
    def df(self):
        return pd.read_json(self.path, lines=True) if self.path.exists() else pd.DataFrame()

def run_parallel(tasks, fn, L, label):
    """Runs fn(task) in parallel for the tasks not yet written; fn returns the record to write."""
    todo = [t for t in tasks if t["key"] not in L.done]
    log.info(f"{label}: {len(todo)} calls ({len(tasks) - len(todo)} already done)")
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = [ex.submit(fn, t) for t in todo]
        for i, f in enumerate(as_completed(futs), 1):
            L.write(f.result())
            if i % 200 == 0: log.info(f"{label}: {i}/{len(todo)}")

def server_timing(L_src, name, label):
    """Queries OpenRouter for each call's server-side time (excluding the network to Kaggle)."""
    L_dst = JsonlLog(name)
    src = L_src.df()
    if src.empty or "gen_id" not in src:
        log.info(f"{label}: no call has an id; server-side time unavailable"); return L_dst
    src = src[(src.status == "ok") & src.gen_id.notna()]
    def fetch(t):
        for attempt in range(8):                      # the metadata can take a few seconds to become available
            try:
                r = session().get(GEN_URL, params={"id": t["gen_id"]}, timeout=30)
                if r.status_code == 200:
                    d = r.json().get("data", {}) or {}
                    prov = (d.get("provider_responses") or [{}])[0] or {}
                    return dict(key=t["key"], gen_id=t["gen_id"], status="ok", server_latency_ms=d.get("latency"),
                                generation_time_ms=d.get("generation_time"), provider_latency_ms=prov.get("latency"),
                                tokens_prompt=d.get("tokens_prompt"), native_tokens_prompt=d.get("native_tokens_prompt"),
                                native_tokens_completion=d.get("native_tokens_completion"), total_cost=d.get("total_cost"))
            except requests.RequestException:
                pass
            time.sleep(2 + attempt)
        return dict(key=t["key"], gen_id=t["gen_id"], status="error")
    run_parallel([dict(key=k, gen_id=g) for k, g in zip(src.key, src.gen_id)], fetch, L_dst, label)
    return L_dst

def humanize(label):
    return re.sub(r"^in:", "", str(label).strip().lower()).replace("_", " ").strip()

def tv(p, q):
    keys = set(p) | set(q)
    return 0.5 * sum(abs(p.get(k, 0) - q.get(k, 0)) for k in keys)

def savefig(name):
    plt.tight_layout(); plt.savefig(OUT / "figures" / f"{name}{SFX}.png", dpi=160); plt.show()

def ask_record(state, qs, gold=None):
    """Calls Jev and returns the common record fields."""
    r = call(state, qs)
    if not r["ok"]:
        return dict(status="error", error=r["error"], attempts=r["attempts"])
    ch, p, conf = answer(r["js"])
    rec = dict(status="ok", pred=ch, probs=p, confidence=conf, latency_ms=r["ms"], attempts=r["attempts"], gen_id=r.get("gen_id"),
               max_decimals=max(decimals(v) for v in p.values()), **usage(r["js"]))
    if gold is not None:
        rec.update(correct=ch == gold, p_gold=p.get(gold, 0.0))
    return rec

In [ ]:
# 3. Data: same label pool (banking77 + clinc150 + MASSIVE = 284); Jev accepts up to 255 per question
from datasets import load_dataset
b77 = load_dataset("mteb/banking77", split="test").to_pandas()[["text", "label_text"]]
B77_LABELS = sorted(b77["label_text"].unique())
POOL, seen, POOL_SOURCES = [], set(), {}
def add_to_pool(labels, source):
    n0 = len(POOL)
    for lab in labels:
        h = humanize(lab)
        if h and h not in seen and h != "oos":
            seen.add(h); POOL.append((h, source))
    POOL_SOURCES[source] = len(POOL) - n0
add_to_pool(B77_LABELS, "banking77")
add_to_pool(load_dataset("clinc/clinc_oos", "plus", split="test").features["intent"].names, "clinc150")
ds = load_dataset("mteb/amazon_massive_intent", "en", split="test")
add_to_pool(sorted(set(ds["label_text" if "label_text" in ds.column_names else "label"])), "mteb/amazon_massive_intent")
log.info(f"label pool: {len(POOL)} unique, by source: {POOL_SOURCES}")
json.dump({"pool_size": len(POOL), "by_source": POOL_SOURCES, "labels": POOL},
          open(OUT / "results" / "label_pool.json", "w"), indent=1)

INSTR = "What does the customer want? Pick the intent that matches the message."

def option_set(gold_h, n, rng):
    b = [h for h, s in POOL if s == "banking77" and h != gold_h]
    rest = [h for h, s in POOL if s != "banking77"]
    rng.shuffle(b); rng.shuffle(rest)
    opts = [gold_h] + (b + rest)[: n - 1]
    rng.shuffle(opts)
    return opts

def sample_b77(n, seed):
    return b77.sample(n=min(n, len(b77)), random_state=seed).reset_index(drop=True)

In [ ]:
# 4. Experiment 1: scaling with the number of options
# Latency: one call at a time. "floor" = short text and 2 options, repeated: the network + server floor.
# Accuracy: 100 texts per N, in parallel.
if RUN["exp1"]:
    L1, LB = JsonlLog("exp1_scaling"), JsonlLog("exp1_latency_bench")
    ex = sample_b77(CFG["exp1_n_examples"], CFG["seed"])
    benches = [("floor", "Hi.", choice_q({"yes": None, "no": None}, "Is this a greeting?"))]
    for n in CFG["exp1_n_options"]:
        rng = np.random.default_rng(CFG["seed"] + n)
        benches.append((str(n), ex.loc[0, "text"],
                        choice_q({o: None for o in option_set(humanize(ex.loc[0, "label_text"]), n, rng)}, INSTR)))
    for name, s0, qs0 in benches:
        for _ in range(3): call(s0, qs0)                                   # warm-up
        for r in range(CFG["exp1_latency_repeats"]):
            k = f"{name}|{r}"
            if k in LB.done: continue
            LB.write(dict(key=k, bench=name, rep=r, **ask_record(s0, qs0)))
        log.info(f"exp1 latency {name} ok")
    LS = server_timing(LB, "exp1_server_timing", "exp1 server time")   # right after the timed calls

    tasks = []
    for n in CFG["exp1_n_options"]:
        for i, row in ex.iterrows():
            gold = humanize(row["label_text"])
            rng = np.random.default_rng(CFG["seed"] * 1000 + i * 7 + n)
            tasks.append(dict(key=f"{n}|{i}", n=n, idx=i, gold=gold, text=row["text"],
                              qs=choice_q({o: None for o in option_set(gold, n, rng)}, INSTR)))
    run_parallel(tasks, lambda t: dict(key=t["key"], n=t["n"], idx=t["idx"], gold=t["gold"],
                                       **ask_record(t["text"], t["qs"], t["gold"])), L1, "exp1 accuracy")

    d1, db = L1.df(), LB.df()
    ok1 = d1[d1.status == "ok"]
    s1 = d1.groupby("n").agg(fail_rate=("status", lambda s: (s != "ok").mean())).join(
        ok1.groupby("n").agg(accuracy=("correct", "mean"), p_gold=("p_gold", "mean"), input_tokens=("input_tokens", "mean"),
                             lat_parallel_median=("latency_ms", "median"), max_decimals=("max_decimals", "max")))
    dbo = db[(db.status == "ok") & (db.attempts == 1)]
    sb = dbo.groupby("bench").agg(lat_median=("latency_ms", "median"), lat_p90=("latency_ms", lambda x: np.percentile(x, 90)),
                                   input_tokens=("input_tokens", "median"), calls=("latency_ms", "size"))
    ds_ = LS.df()
    if not ds_.empty and "server_latency_ms" in ds_:
        st = dbo.merge(ds_[ds_.status == "ok"][["key", "server_latency_ms", "generation_time_ms", "provider_latency_ms"]], on="key")
        sb = sb.join(st.groupby("bench").agg(server_latency_median=("server_latency_ms", "median"),
                                              generation_time_median=("generation_time_ms", "median"),
                                              provider_latency_median=("provider_latency_ms", "median")))
    floor = sb.loc["floor", "lat_median"] if "floor" in sb.index else np.nan
    sb["lat_minus_floor"] = sb.lat_median - floor
    s1 = s1.join(sb.drop(index="floor", errors="ignore").rename(index=int), how="left", rsuffix="_bench")
    s1.insert(0, "regime", "jev"); s1.insert(1, "n_eff", s1.index)
    s1.to_csv(OUT / "results" / f"exp1_summary{SFX}.csv"); sb.to_csv(OUT / "results" / f"exp1_latency{SFX}.csv")
    display(s1); display(sb)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(s1.index, s1.lat_median, "o-", label="median"); ax[0].axhline(floor, ls="--", c="gray", label="floor (2 options, short text)")
    ax[1].plot(s1.index, s1.accuracy, "o-"); ax[2].plot(s1.index, s1.input_tokens, "o-")
    for a_, t in zip(ax, ["latency (ms, including network)", "accuracy", "input tokens charged"]):
        a_.set_xscale("log"); a_.set_xlabel("number of options"); a_.set_title(t)
    ax[0].legend(); savefig("exp1_scaling")

In [ ]:
# 5. Experiment 2: invariance to option order, with a repetition control
# perm 0 = original order; perm 1..P-1 = permutations; "rep" = the original order sent again.
# The flip rate between the two identical calls is the non-determinism floor.
if RUN["exp2"]:
    L2 = JsonlLog("exp2_order")
    ex = sample_b77(CFG["exp2_n_examples"], CFG["seed"] + 1)
    tasks = []
    for n in CFG["exp2_n_options"]:
        for i, row in ex.iterrows():
            gold = humanize(row["label_text"])
            rng = np.random.default_rng(CFG["seed"] * 2000 + i * 11 + n)
            base_opts = option_set(gold, n, rng)
            orders = [list(base_opts)] + [list(rng.permutation(base_opts)) for _ in range(CFG["exp2_n_perms"] - 1)]
            for pi, order in list(enumerate(orders)) + [("rep", orders[0])]:
                tasks.append(dict(key=f"{n}|{i}|{pi}", n=n, idx=i, perm=pi, gold=gold, text=row["text"],
                                  gold_pos=order.index(gold), qs=choice_q({o: None for o in order}, INSTR)))
    run_parallel(tasks, lambda t: dict(key=t["key"], n=t["n"], idx=t["idx"], perm=str(t["perm"]), gold=t["gold"],
                                       gold_pos=t["gold_pos"], **ask_record(t["text"], t["qs"], t["gold"])), L2, "exp2")

    d2 = L2.df(); d2 = d2[d2.status == "ok"]; d2["perm"] = d2.perm.astype(str)
    rows = []
    for (n, i), g in d2.groupby(["n", "idx"]):
        perms = g[g.perm != "rep"]; rep = g[g.perm == "rep"]; orig = g[g.perm == "0"]
        P = list(perms.probs)
        row = dict(n=n, idx=i, flipped=perms.pred.nunique() > 1,
                   mean_tv=np.mean([tv(P[a], P[b]) for a in range(len(P)) for b in range(a + 1, len(P))]) if len(P) > 1 else np.nan,
                   p_gold_std=perms.p_gold.std(), accuracy=perms.correct.mean())
        if len(rep) and len(orig):
            row.update(repeat_flipped=rep.pred.iloc[0] != orig.pred.iloc[0], repeat_tv=tv(rep.probs.iloc[0], orig.probs.iloc[0]))
        rows.append(row)
    s2 = pd.DataFrame(rows).groupby("n").agg(flip_rate=("flipped", "mean"), mean_tv=("mean_tv", "mean"),
                                             p_gold_std=("p_gold_std", "mean"), accuracy=("accuracy", "mean"),
                                             repeat_flip_rate=("repeat_flipped", "mean"), repeat_tv=("repeat_tv", "mean"))
    s2.insert(0, "regime", "jev")
    p2 = d2[d2.perm != "rep"].copy()
    p2["pos_bin"] = pd.cut(p2.gold_pos / (p2.n - 1).clip(lower=1), [-0.01, 0.25, 0.5, 0.75, 1.0],
                           labels=["first", "second", "third", "last"])
    pos = p2.groupby(["n", "pos_bin"], observed=True).correct.mean().unstack()
    pos.insert(0, "regime", "jev")
    s2.to_csv(OUT / "results" / f"exp2_summary{SFX}.csv"); pos.to_csv(OUT / "results" / f"exp2_position_bias{SFX}.csv")
    display(s2); display(pos)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(s2.index, s2.flip_rate, "o-", label="different orders"); ax[0].plot(s2.index, s2.repeat_flip_rate, "o--", label="same order, 2 calls")
    ax[1].plot(s2.index, s2.mean_tv, "o-", label="different orders"); ax[1].plot(s2.index, s2.repeat_tv, "o--", label="same order, 2 calls")
    for a_, t in zip(ax, ["fraction of texts whose answer changes", "mean distance between distributions (TV)"]):
        a_.set_xlabel("number of options"); a_.set_title(t); a_.legend()
    savefig("exp2_order")

In [ ]:
# 6. Experiment 2b: position in the list or length of the options listed after the correct one?
# Without access to the model, the "distance" is the text of the options listed after the correct one:
# whether the options come before the state (as in Laya) or before the answer position (as in Lev),
# this span separates the correct option from the point that matters. The long layout lengthens it without changing the position.
if RUN["exp2b"]:
    import statsmodels.formula.api as smf
    N2B, POS2B = 50, [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 49]
    LAYOUTS = {"short": None, "long": "banking intent"}
    L2b = JsonlLog("exp2b_distance")
    ex = sample_b77(CFG["exp2b_n_examples"], CFG["seed"] + 21)
    tasks = []
    for i, row in ex.iterrows():
        gold = humanize(row["label_text"])
        rng = np.random.default_rng(CFG["seed"] * 5000 + i)
        distract = [o for o in option_set(gold, N2B, rng) if o != gold]
        for layout, desc in LAYOUTS.items():
            for p in POS2B:
                order = distract[:p] + [gold] + distract[p:]
                after = order[p + 1:]
                chars_after = sum(len(o) + (len(desc) + 2 if desc else 0) + 1 for o in after)
                tasks.append(dict(key=f"{layout}|{p}|{i}", layout=layout, pos=p, idx=i, gold=gold, text=row["text"],
                                  chars_after=chars_after, qs=choice_q({o: desc for o in order}, INSTR)))
    run_parallel(tasks, lambda t: dict(key=t["key"], layout=t["layout"], pos=t["pos"], idx=t["idx"], gold=t["gold"],
                                       chars_after=t["chars_after"], **ask_record(t["text"], t["qs"], t["gold"])), L2b, "exp2b")
    d = L2b.df(); d = d[d.status == "ok"]
    d["rank_gold"] = [sorted(p.values(), reverse=True).index(p[g]) + 1 for p, g in zip(d.probs, d.gold)]
    s_pos = d.groupby(["layout", "pos"]).agg(chars_after=("chars_after", "mean"), accuracy=("correct", "mean"),
                                             p_gold=("p_gold", "mean"), rank_gold=("rank_gold", "mean"),
                                             input_tokens=("input_tokens", "mean"))
    d["y"] = d.correct.astype(int); d["pos_std"] = d.pos / 49; d["dist_std"] = d.chars_after / 400
    try:
        fit = smf.logit("y ~ pos_std + dist_std", data=d).fit(disp=0, cov_type="cluster", cov_kwds={"groups": d.idx})
        coefs = pd.DataFrame({"coef": fit.params, "p_value": fit.pvalues})
    except Exception as e:
        log.info(f"exp2b: logistic regression failed ({e!r})"); coefs = pd.DataFrame()
    try:   # the probability of the correct option varies even when accuracy is at the ceiling
        fit_p = smf.ols("p_gold ~ pos_std + dist_std", data=d).fit(cov_type="cluster", cov_kwds={"groups": d.idx})
        coefs_p = pd.DataFrame({"coef": fit_p.params, "p_value": fit_p.pvalues})
    except Exception as e:
        log.info(f"exp2b: linear regression failed ({e!r})"); coefs_p = pd.DataFrame()
    s_pos.to_csv(OUT / "results" / f"exp2b_by_position{SFX}.csv")
    coefs.to_csv(OUT / "results" / f"exp2b_logit{SFX}.csv"); coefs_p.to_csv(OUT / "results" / f"exp2b_ols_pgold{SFX}.csv")
    display(s_pos); display(coefs); display(coefs_p)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for layout, g in s_pos.reset_index().groupby("layout"):
        ax[0].plot(g.pos, g.p_gold, "o-", label=layout); ax[1].plot(g.chars_after, g.p_gold, "o-", label=layout)
    ax[0].set_xlabel("position of the correct option (0 = first)"); ax[1].set_xlabel("characters of the options after the correct one")
    for a_ in ax: a_.set_ylabel("probability of the correct option"); a_.set_ylim(0, 1); a_.legend()
    savefig("exp2b_distance")

In [ ]:
# 7. Experiment 3b: minimal-edit pairs (same set as the other models)
T3B = [
    ("count", "change", "I was charged once for my {0}.", "I was charged twice for my {0}.",
     ("single charge", "a question about one normal charge"), ("duplicate charge", "the same purchase was billed more than once"),
     ("A", "B"), [("coffee",), ("taxi ride",), ("hotel booking",)]),
    ("count", "change", "I entered the wrong PIN once {0}.", "I entered the wrong PIN three times {0}.",
     ("pin question", "a general question about the PIN"), ("card blocked", "the card is blocked after too many wrong PIN attempts"),
     ("A", "B"), [("at the ATM",), ("at a shop",), ("at the petrol station",)]),
    ("count", "change", "I paid for one {0} and received one.", "I paid for two {1} and received one.",
     ("order complete", "everything that was paid for was delivered"), ("missing item", "fewer items were delivered than were paid for"),
     ("A", "B"), [("ticket", "tickets"), ("shirt", "shirts"), ("book", "books")]),
    ("count", "change", "My {0} was paid into my account once this month.", "My {0} was paid into my account twice this month.",
     ("normal payment", "an expected single payment was received"), ("duplicate payment", "the same payment was received more than once"),
     ("A", "B"), [("salary",), ("pension",), ("allowance",)]),
    ("negation", "change", "I want to cancel my {0}.", "I do not want to cancel my {0}.",
     ("cancel", "the customer wants to end it"), ("keep", "the customer wants to keep it"),
     ("A", "B"), [("subscription",), ("membership",), ("plan",)]),
    ("negation", "change", "The {0} arrived.", "The {0} never arrived.",
     ("delivered", "the item was delivered"), ("delivery problem", "the item has not been delivered"),
     ("A", "B"), [("package",), ("card",), ("order",)]),
    ("negation", "change", "I recognise this {0} on my statement.", "I don't recognise this {0} on my statement.",
     ("confirmed", "the customer confirms it is theirs"), ("fraud report", "the customer did not make it"),
     ("A", "B"), [("payment",), ("transfer",), ("charge",)]),
    ("negation", "change", "The {0} works after the update.", "The {0} no longer works after the update.",
     ("working fine", "it works as expected"), ("bug", "it is broken or not working"),
     ("A", "B"), [("app",), ("website",), ("card reader",)]),
    ("numbers", "change", "My balance is ${0} and I want to withdraw ${1}.", "My balance is ${1} and I want to withdraw ${0}.",
     ("sufficient funds", "the balance covers the withdrawal"), ("insufficient funds", "the withdrawal is larger than the balance"),
     ("A", "B"), [("500", "200"), ("900", "300"), ("1000", "50")]),
    ("numbers", "change", "The invoice says ${0} and I was charged ${0}.", "The invoice says ${0} and I was charged ${1}.",
     ("amount correct", "charged exactly the invoice amount"), ("amount mismatch", "charged a different amount than the invoice"),
     ("A", "B"), [("120", "210"), ("45", "54"), ("310", "130")]),
    ("numbers", "change", "I ordered size {0} and received size {0}.", "I ordered size {0} and received size {1}.",
     ("correct item", "received exactly the size ordered"), ("wrong item", "received a different size than ordered"),
     ("A", "B"), [("42", "24"), ("38", "83"), ("46", "64")]),
    ("numbers", "change", "My flight leaves at {0} and I will get to the airport at {1}.", "My flight leaves at {1} and I will get to the airport at {0}.",
     ("on time", "the customer will make the flight"), ("missed flight", "the customer will miss the flight"),
     ("A", "B"), [("9", "7"), ("11", "8"), ("6", "4")]),
    ("roles", "change", "{1} sent money to me.", "I sent money to {0}.",
     ("incoming transfer", "the customer received money"), ("outgoing transfer", "the customer sent money"),
     ("A", "B"), [("the bank", "The bank"), ("my landlord", "My landlord"), ("my friend", "My friend")]),
    ("roles", "change", "I refunded the {0}.", "The {0} refunded me.",
     ("refund issued", "the writer gave money back"), ("refund received", "the writer got money back"),
     ("A", "B"), [("customer",), ("shop",), ("seller",)]),
    ("roles", "change", "I owe {0} money.", "{1} owes me money.",
     ("writer owes", "the writer has a debt to someone"), ("owed to writer", "someone has a debt to the writer"),
     ("A", "B"), [("my brother", "My brother"), ("the shop", "The shop"), ("my landlord", "My landlord")]),
    ("roles", "change", "{1} paid me.", "I paid {0}.",
     ("money received", "the writer received a payment"), ("money paid", "the writer made a payment"),
     ("A", "B"), [("my employer", "My employer"), ("my tenant", "My tenant"), ("my client", "My client")]),
    ("time", "change", "I want to pay before {0}.", "I want to pay after {0}.",
     ("on-time payment", "paying before the deadline"), ("late payment", "paying after the deadline"),
     ("A", "B"), [("the due date",), ("the deadline",), ("the end of the month",)]),
    ("time", "change", "I will travel {0} next week.", "I travelled {0} last week.",
     ("future trip", "about a trip that has not happened yet"), ("past trip", "about a trip that already happened"),
     ("A", "B"), [("abroad",), ("to Spain",), ("to Japan",)]),
    ("time", "change", "I want to stop a {0} that will go out tomorrow.", "I want to stop a {0} that went out yesterday.",
     ("stop future payment", "cancel a payment before it is sent"), ("reverse past payment", "get back a payment that was already sent"),
     ("A", "B"), [("payment",), ("transfer",), ("direct debit",)]),
    ("time", "change", "My {0} renews next month.", "My {0} renewed last month.",
     ("upcoming renewal", "a renewal that has not happened yet"), ("past renewal", "a renewal that already happened"),
     ("A", "B"), [("subscription",), ("membership",), ("insurance",)]),
    ("quantifier", "change", "All of my {0} failed today.", "None of my {0} failed today.",
     ("failed", "they did not go through"), ("went through", "they went through normally"),
     ("A", "B"), [("transfers",), ("payments",), ("top-ups",)]),
    ("quantifier", "change", "Every {0} I tried was declined.", "No {0} I tried was declined.",
     ("declined", "the attempts were rejected"), ("accepted", "the attempts were accepted"),
     ("A", "B"), [("card payment",), ("withdrawal",), ("transfer",)]),
    ("quantifier", "change", "All of my {0} were refunded.", "None of my {0} were refunded.",
     ("refund complete", "the money was returned"), ("refund missing", "the money was not returned"),
     ("A", "B"), [("orders",), ("purchases",), ("tickets",)]),
    ("quantifier", "change", "I received all of my {0}.", "I received none of my {0}.",
     ("all delivered", "everything arrived"), ("nothing delivered", "nothing arrived"),
     ("A", "B"), [("parcels",), ("cards",), ("documents",)]),
    # controls: the edit does not change the meaning, so the answer should not change
    ("paraphrase", "same", "I was charged twice for my {0}.", "I was charged two times for my {0}.",
     ("single charge", "a question about one normal charge"), ("duplicate charge", "the same purchase was billed more than once"),
     ("B", "B"), [("coffee",), ("taxi ride",), ("hotel booking",)]),
    ("paraphrase", "same", "I do not want to cancel my {0}.", "I don't want to cancel my {0}.",
     ("cancel", "the customer wants to end it"), ("keep", "the customer wants to keep it"),
     ("B", "B"), [("subscription",), ("membership",), ("plan",)]),
    ("paraphrase", "same", "My balance is ${1} and I want to withdraw ${0}.", "My balance is {1} dollars and I want to withdraw {0} dollars.",
     ("sufficient funds", "the balance covers the withdrawal"), ("insufficient funds", "the withdrawal is larger than the balance"),
     ("B", "B"), [("500", "200"), ("900", "300"), ("1000", "50")]),
    ("paraphrase", "same", "The {0} never arrived.", "The {0} has not arrived.",
     ("delivered", "the item was delivered"), ("delivery problem", "the item has not been delivered"),
     ("B", "B"), [("package",), ("card",), ("order",)]),
    ("paraphrase", "same", "I sent money to {0}.", "I transferred money to {0}.",
     ("incoming transfer", "the customer received money"), ("outgoing transfer", "the customer sent money"),
     ("B", "B"), [("the bank",), ("my landlord",), ("my friend",)]),
]
OTHER3B, INSTR3B = ("other", "anything else"), "Which option best describes the customer's message?"
if RUN["exp3b"]:
    L3b = JsonlLog("exp3b_minimal_pairs")
    def do_pair(t):
        ra = ask_record(t["ta"], t["qs"]); rb = ask_record(t["tb"], t["qs"])
        rec = dict(key=t["key"], template=t["template"], filler=t["filler"], category=t["category"], kind=t["kind"],
                   order=t["order"], text_a=t["ta"], text_b=t["tb"], gold_a=t["gA"], gold_b=t["gB"])
        if ra["status"] != "ok" or rb["status"] != "ok":
            return {**rec, "status": "error", "error": ra.get("error") or rb.get("error")}
        PA, PB = ra["probs"], rb["probs"]
        return {**rec, "status": "ok", "cost": (ra.get("cost") or 0) + (rb.get("cost") or 0),
                "pred_a": ra["pred"], "pred_b": rb["pred"], "correct_a": ra["pred"] == t["gA"],
                "correct_b": rb["pred"] == t["gB"], "flipped": ra["pred"] != rb["pred"], "tv": tv(PA, PB),
                "p_gold_b_in_a": PA.get(t["gB"], 0.0), "p_gold_b_in_b": PB.get(t["gB"], 0.0), "probs_a": PA, "probs_b": PB}
    tasks = []
    for t_id, (cat, kind, ta_t, tb_t, oA, oB, (ga, gb), fills) in enumerate(T3B):
        opt = {"A": oA, "B": oB}
        for f_id, f in enumerate(fills):
            for order_name, seq in [("fwd", ["A", "B"]), ("rev", ["B", "A"])]:
                crit = {opt[s][0]: opt[s][1] for s in seq}; crit[OTHER3B[0]] = OTHER3B[1]
                tasks.append(dict(key=f"{t_id}|{f_id}|{order_name}", template=t_id, filler=f_id, category=cat, kind=kind,
                                  order=order_name, ta=ta_t.format(*f), tb=tb_t.format(*f), gA=opt[ga][0], gB=opt[gb][0],
                                  qs=choice_q(crit, INSTR3B)))
    run_parallel(tasks, do_pair, L3b, "exp3b")
    d = L3b.df(); d = d[d.status == "ok"]
    d["both_correct"] = d.correct_a & d.correct_b
    d["shift_to_gold_b"] = d.p_gold_b_in_b - d.p_gold_b_in_a
    d["moved_right_way"] = d.shift_to_gold_b > 0
    agg = dict(runs=("key", "count"), acc_a=("correct_a", "mean"), acc_b=("correct_b", "mean"),
               both_correct=("both_correct", "mean"), flip_rate=("flipped", "mean"), mean_tv=("tv", "mean"),
               shift_to_gold_b=("shift_to_gold_b", "mean"), moved_right_way=("moved_right_way", "mean"))
    s3b = d.groupby(["kind", "category"]).agg(**agg)
    s3b_t = d.groupby(["kind", "category", "template"]).agg(**agg)
    s3b_t["example_a"] = d.groupby(["kind", "category", "template"]).text_a.first()
    s3b.to_csv(OUT / "results" / f"exp3b_summary{SFX}.csv"); s3b_t.to_csv(OUT / "results" / f"exp3b_by_template{SFX}.csv")
    display(s3b)

In [ ]:
# 8. Experiment 4: labels without meaning (same 12 intents and conditions)
INTENTS4 = {
    "activate_my_card": "how to activate a newly received card",
    "age_limit": "minimum age required to open or use an account",
    "atm_support": "which cash machines can be used and how",
    "card_swallowed": "a cash machine kept or retained the card",
    "change_pin": "how to change the card PIN code",
    "exchange_rate": "what currency exchange rate is applied",
    "lost_or_stolen_phone": "the phone with the banking app was lost or stolen",
    "terminate_account": "close or delete the account permanently",
    "top_up_limits": "maximum amount that can be added to the account",
    "transaction_charged_twice": "the same payment appears twice",
    "request_refund": "ask for money back for a purchase",
    "edit_personal_details": "update name, address or other personal information",
}
missing = [k for k in INTENTS4 if k not in B77_LABELS]
if missing: log.info(f"exp4: intents missing from banking77, removed: {missing}")
INTENTS4 = {k: v for k, v in INTENTS4.items() if k in B77_LABELS}
I4 = list(INTENTS4)
rng4 = np.random.default_rng(CFG["seed"] + 4)
CODES = {lab: f"K{c}" for lab, c in zip(I4, rng4.choice(np.arange(10, 100), len(I4), replace=False))}
shift = rng4.integers(1, len(I4))
MISLEAD = {lab: humanize(I4[(j + shift) % len(I4)]) for j, lab in enumerate(I4)}  # no key points to its own intent

def criteria_for(cond, order):
    """Returns {shown_key: description} and {shown_key: true intent of the description}."""
    crit, key2int = {}, {}
    for lab in order:
        key = dict(original=humanize(lab), name_only=humanize(lab), code_desc=CODES[lab],
                   code_only=CODES[lab], misleading=MISLEAD[lab])[cond]
        crit[key] = None if cond in ("name_only", "code_only") else INTENTS4[lab]
        key2int[key] = lab
    return crit, key2int
INSTR4 = "Which option matches the customer's request?"
if RUN["exp4"]:
    L4 = JsonlLog("exp4_labels")
    # same sample as the other notebooks (groupby + sample per intent), independent of the pandas version
    sub = pd.concat([g.sample(n=min(CFG["exp4_per_intent"], len(g)), random_state=CFG["seed"])
                     for _, g in b77[b77.label_text.isin(I4)].groupby("label_text")]).reset_index(drop=True)
    tasks = []
    for i, row in sub.iterrows():
        order = list(np.random.default_rng(CFG["seed"] * 4000 + i).permutation(I4))
        for cond in ["original", "name_only", "code_desc", "code_only", "misleading"]:
            crit, key2int = criteria_for(cond, order)
            tasks.append(dict(key=f"{i}|{cond}", idx=i, cond=cond, gold=row["label_text"], text=row["text"],
                              key2int=key2int, qs=choice_q(crit, INSTR4)))
    def do4(t):
        r = ask_record(t["text"], t["qs"])
        rec = dict(key=t["key"], idx=t["idx"], condition=t["cond"], gold=t["gold"])
        if r["status"] != "ok":
            return {**rec, **r}
        pred_int = t["key2int"][r["pred"]]
        return {**rec, "status": "ok", "pred_key": r["pred"], "pred_intent": pred_int, "correct": pred_int == t["gold"],
                "followed_key": t["cond"] == "misleading" and r["pred"] == humanize(t["gold"]),
                "p_max": max(r["probs"].values()), "input_tokens": r["input_tokens"], "cost": r.get("cost")}
    run_parallel(tasks, do4, L4, "exp4")
    d4 = L4.df(); d4 = d4[d4.status == "ok"]
    s4 = d4.groupby("condition").agg(n=("idx", "count"), accuracy=("correct", "mean"), followed_key=("followed_key", "mean"),
                                     p_max=("p_max", "mean"), input_tokens=("input_tokens", "mean"))
    s4 = s4.reindex(["original", "name_only", "code_desc", "code_only", "misleading"]); s4["chance"] = 1 / len(I4)
    s4.to_csv(OUT / "results" / f"exp4_summary{SFX}.csv"); display(s4)

In [ ]:
# 9. Experiment R: what Jev reads, and how many times (token counts and latency)
# Each factor varies alone; calls are made one at a time so that latency is also valid.
#   R1 number of options · R2 description length · R3 state length · R4 number of questions (short and long state)
# In R4, if the state is charged once per question, the slope with the long state exceeds the slope with the short state
# by the length difference between the two states; if it is charged only once, the two slopes coincide.
# (R2 uses at most 26 words because the filler text has 26 words.)
if RUN["expR"]:
    LR = JsonlLog("expR_tokens")
    SHORT = "My card was charged twice for the same coffee."
    LONG_PARTS = sample_b77(8, CFG["seed"] + 77).text.tolist()
    FILL = ("this option describes a banking customer request about accounts cards payments transfers "
            "fees limits verification security and other services offered to customers of the bank today").split()
    NAMES = [humanize(l) for l in I4]
    configs = []
    for n in [2, 4, 8, 16, 32, 64, 128, 255]:
        configs.append(("R1_options", n, SHORT, choice_q({h: None for h, _ in POOL[:n]}, INSTR)))
    for k in [0, 5, 10, 20, 26]:
        configs.append(("R2_desc_words", k, SHORT, choice_q({h: (" ".join(FILL[:k]) if k else None) for h in NAMES}, INSTR)))
    for m in [1, 2, 4, 8]:
        configs.append(("R3_state_sentences", m, " ".join(LONG_PARTS[:m]), choice_q({h: None for h in NAMES}, INSTR)))
    for label, st in [("R4_questions_short_state", SHORT), ("R4_questions_long_state", " ".join(LONG_PARTS))]:
        for q in [1, 2, 4, 8]:
            qs = {f"q{j}": {"type": "choice", "instructions": f"Question {j + 1}: {INSTR}",
                            "criteria": {h: None for h in NAMES[:5]}} for j in range(q)}
            configs.append((label, q, st, qs))
    for factor, level, st, qs in configs:
        call(st, qs)                                                           # warm-up
        for r in range(CFG["expR_repeats"]):
            k = f"{factor}|{level}|{r}"
            if k in LR.done: continue
            res = call(st, qs)
            rec = dict(key=k, factor=factor, level=level, rep=r, state_chars=len(st), n_questions=len(qs))
            rec.update(dict(status="ok", latency_ms=res["ms"], attempts=res["attempts"], gen_id=res.get("gen_id"), **usage(res["js"]))
                       if res["ok"] else dict(status="error", error=res["error"]))
            LR.write(rec)
        log.info(f"expR {factor}={level} ok")

    LRS = server_timing(LR, "expR_server_timing", "expR server time")
    dR = LR.df(); dR = dR[(dR.status == "ok") & (dR.attempts == 1)]
    dRS = LRS.df()
    if not dRS.empty and "server_latency_ms" in dRS:
        dR = dR.merge(dRS[dRS.status == "ok"][["key", "server_latency_ms", "generation_time_ms", "native_tokens_prompt"]],
                      on="key", how="left")
    for c in ["server_latency_ms", "generation_time_ms", "native_tokens_prompt"]:
        if c not in dR: dR[c] = np.nan
    sR = dR.groupby(["factor", "level"]).agg(input_tokens=("input_tokens", "median"), output_tokens=("output_tokens", "median"),
                                             tokens_min=("input_tokens", "min"), tokens_max=("input_tokens", "max"),
                                             lat_median=("latency_ms", "median"), server_latency=("server_latency_ms", "median"),
                                             generation_time=("generation_time_ms", "median"),
                                             native_tokens_prompt=("native_tokens_prompt", "median"),
                                             cost=("cost", "median"), calls=("latency_ms", "size"))
    sR.to_csv(OUT / "results" / f"expR_summary{SFX}.csv"); display(sR)

    def slope(factor, col="input_tokens"):
        g = sR.loc[factor].reset_index()[["level", col]].dropna()
        return np.polyfit(g.level.astype(float), g[col].astype(float), 1) if len(g) > 1 else (np.nan, np.nan)
    fits = {f: dict(tokens_per_unit=slope(f)[0], tokens_intercept=slope(f)[1], ms_per_unit=slope(f, "lat_median")[0],
                    server_ms_per_unit=slope(f, "server_latency")[0] if sR["server_latency"].notna().any() else np.nan)
            for f in sR.index.get_level_values(0).unique()}
    fits = pd.DataFrame(fits).T
    try:   # how many more tokens the long state has than the short one, measured with 1 question
        extra_state = (sR.loc[("R4_questions_long_state", 1), "input_tokens"] - sR.loc[("R4_questions_short_state", 1), "input_tokens"])
        extra_slope = fits.loc["R4_questions_long_state", "tokens_per_unit"] - fits.loc["R4_questions_short_state", "tokens_per_unit"]
        fits.loc["state_counted_per_question", "tokens_per_unit"] = extra_slope / extra_state if extra_state else np.nan
        log.info(f"expR: state charged per question = {extra_slope / extra_state:.2f} (≈1: re-read for every question; ≈0: read once)")
    except Exception as e:
        log.info(f"expR: state comparison failed ({e!r})")
    fits.to_csv(OUT / "results" / f"expR_fits{SFX}.csv"); display(fits)

    fig, ax = plt.subplots(1, 4, figsize=(18, 4))
    for a_, f in zip(ax, ["R1_options", "R2_desc_words", "R3_state_sentences", "R4_questions_short_state"]):
        g = sR.loc[f].reset_index(); a_.plot(g.level, g.input_tokens, "o-", label=f.split("_", 1)[1])
        if f.startswith("R4"):
            g2 = sR.loc["R4_questions_long_state"].reset_index(); a_.plot(g2.level, g2.input_tokens, "o--", label="questions, long state")
        a_.set_title(f); a_.set_ylabel("input tokens charged"); a_.legend()
        a2 = a_.twinx(); a2.plot(g.level, g.lat_median, "s:", c="gray"); a2.set_ylabel("latency (ms)", color="gray")
    savefig("expR_tokens")

In [ ]:
# 10. Total cost and packaging
# Reads the JSONL files line by line (without pandas), so as not to depend on how pandas parses each column.
costs = []
for f in sorted((OUT / "results").glob("*.jsonl")):
    if QUICK != ("_quick" in f.name):
        continue
    total, n_calls, bad = 0.0, 0, 0
    try:
        with f.open() as fh:
            for line in fh:
                try:
                    c = json.loads(line).get("cost")
                    total += float(c) if isinstance(c, (int, float)) else 0.0
                    n_calls += 1
                except Exception:
                    bad += 1
    except Exception as e:
        log.info(f"cost: could not read {f.name} ({e!r})")
    costs.append((f.name, total, n_calls, bad))
cost_df = pd.DataFrame(costs, columns=["file", "cost_usd", "records", "invalid_lines"])
print(cost_df.to_string(index=False))
log.info(f"total cost reported by OpenRouter: US$ {cost_df.cost_usd.sum():.4f}")

zpath = ROOT / f"jev_bench_{RUN_ID}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(ROOT))
log.info(f"done: {zpath}")